# P01 (basic) — discrete-event systems: finite automata, reachability & composition

**Module 23 — Advanced Automation**

The discrete layer of automation asks: *what state is the plant in, and which event triggers which
action?* The tool is the **finite automaton**. You model an automated **bottle-filling station** as
two automata (a fill valve and a bottle position), **simulate** event sequences, compute the
**reachable** states, and **compose** the two components into one model. The punchline is a
**safety** finding: the uncontrolled system can reach a forbidden "valve open with no bottle" state
— which is exactly what the supervisor of P02 will prevent.

### Goal
- represent a **deterministic finite automaton (DFA)** $G=(Q,\Sigma,\delta,q_0,Q_m)$ (script ch. 3),
- **simulate** it (run an event string) and compute the **reachable set** by breadth-first search,
- build the **parallel composition** (synchronous product) of two components, with shared events synchronising,
- verify a **safety property** by checking a forbidden state is (un)reachable.

### Format
Jupyter notebook — automata are best understood by defining one, running it, and inspecting the
reachable state set.

### Prior knowledge
The module 23 script ch. 1-3, graph search / BFS (module 06), basic set operations.

### Tasks
Much is given; at the `# TODO` spots you implement the simulator, the reachability search and the composition rule. The solution is in `solution/`.

## Setup
No numerical libraries needed — automata are sets, dictionaries and graph search. Plain Python.

In [ ]:
from collections import deque

# We represent a DFA as a dict:
#   states : set of state names
#   events : the event alphabet (Sigma)
#   trans  : dict {(state, event): next_state}   (the partial transition function delta)
#   init   : the initial state q0
#   marked : set of marked (task-complete) states
def automaton(states, events, trans, init, marked):
    return dict(states=set(states), events=set(events), trans=dict(trans),
                init=init, marked=set(marked))

## Part A — model the two components (given)

Our automated bottle-filling station has two components:

- **Valve** (the filler): states `closed`/`open`; events `open_valve`, `close_valve` (both
  controllable — the automation commands them), plus a shared event `depart`.
- **Bottle** (the position at the station): states `no_bottle`/`bottle`; events `arrive` (a bottle
  arrives on the conveyor — *uncontrollable*) and `depart` (a filled bottle leaves — controllable).

The event **`depart` is shared**: a bottle may only leave when the **valve is closed** (you don't
move a bottle out from under an open valve). We encode that by giving the valve a `depart` self-loop
*only* in the `closed` state — so in the composition, `depart` synchronises the two.

In [ ]:
valve = automaton(
    states=["closed", "open"],
    events=["open_valve", "close_valve", "depart"],
    trans={("closed", "open_valve"): "open",
           ("open", "close_valve"): "closed",
           ("closed", "depart"): "closed"},      # depart allowed only while closed
    init="closed", marked=["closed"])

bottle = automaton(
    states=["no_bottle", "bottle"],
    events=["arrive", "depart"],
    trans={("no_bottle", "arrive"): "bottle",
           ("bottle", "depart"): "no_bottle"},
    init="no_bottle", marked=["no_bottle"])

# controllable vs. uncontrollable events (needed in P02; stated here for completeness)
UNCONTROLLABLE = {"arrive"}          # a bottle arriving cannot be prevented
print("valve events:", valve["events"])
print("bottle events:", bottle["events"])
print("shared events:", valve["events"] & bottle["events"])

## Part B — simulate the automaton

Running a DFA means following $\delta$ along an event string. **Your task:** implement `run(G, string)`
that starts at `G["init"]` and applies each event via `G["trans"]`; return the final state, or `None`
if some event is **undefined** in the current state (the string is not in the language).

In [ ]:
def run(G, string):
    """Follow delta from the initial state along the event string.
    Return the final state, or None if an event is undefined on the way."""
    state = G["init"]
    # TODO: for each event, look up (state, event) in G["trans"];
    #       if missing, return None; otherwise advance. Return the final state.
    raise NotImplementedError

# a legal sequence: open, close, arrive, depart -> back to closed
print("valve after [open_valve, close_valve]:", run(valve, ["open_valve", "close_valve"]))
# an illegal sequence: close_valve is undefined in the initial 'closed' state -> None
print("valve after [close_valve]:", run(valve, ["close_valve"]))
# bottle: arrive then depart returns to no_bottle
print("bottle after [arrive, depart]:", run(bottle, ["arrive", "depart"]))

**Expectation.** `[open_valve, close_valve]` returns `closed`; `[close_valve]` from the initial
`closed` state returns `None` (undefined — you cannot close an already-closed valve); `[arrive,
depart]` returns `no_bottle`.

## Part C — reachability

Which states can actually occur? **Your task:** implement `reachable(G)` — a breadth-first search
from `q_0` following every defined transition, returning the set of reachable states (script ch. 3;
this is the graph search of module 06).

In [ ]:
def reachable(G):
    """Breadth-first search from the initial state; return the set of reachable states."""
    seen = {G["init"]}
    queue = deque([G["init"]])
    # TODO: pop a state, try every event; for each defined (state, event),
    #       enqueue the successor if not already seen. Return `seen`.
    raise NotImplementedError

print("valve reachable:", reachable(valve))
print("bottle reachable:", reachable(bottle))

**Expectation.** Both components have all their states reachable (`{closed, open}` and
`{no_bottle, bottle}`). Reachability is how you later prove a *bad* state can never occur — by
showing it is **not** in this set.

## Part D — parallel composition (the synchronous product)

Real plants are several components at once. The **synchronous product** $G_1\|G_2$ runs on the joint
state set $Q_1\times Q_2$ with the rule:
- a **shared** event (in both alphabets) fires only if **both** components can take it — they
  *synchronise*, and both move;
- a **private** event (in one alphabet only) fires whenever its owner can take it; the other
  component stays put.

**Your task:** fill in the transition rule inside `compose`. For a product state $(s_1,s_2)$ and an
event $e$, decide whether $e$ is enabled and what the successor is, following the two bullets above.

In [ ]:
def compose(G1, G2):
    """Synchronous product G1 || G2. Shared events synchronise; private events interleave."""
    shared = G1["events"] & G2["events"]
    all_events = G1["events"] | G2["events"]
    init = (G1["init"], G2["init"])
    trans = {}
    seen = {init}
    queue = deque([init])
    while queue:
        (s1, s2) = queue.popleft()
        for e in all_events:
            nxt = None
            # TODO: set `nxt` = the successor product state, or leave it None if e is disabled.
            #   shared event (e in shared): enabled iff (s1,e) in G1.trans AND (s2,e) in G2.trans;
            #                               successor = (delta1(s1,e), delta2(s2,e))
            #   e private to G1 (e in G1.events only): enabled iff (s1,e) in G1.trans;
            #                               successor = (delta1(s1,e), s2)
            #   e private to G2: symmetric.
            raise NotImplementedError
            if nxt is not None:
                trans[((s1, s2), e)] = nxt
                if nxt not in seen:
                    seen.add(nxt)
                    queue.append(nxt)
    marked = {(a, b) for a in G1["marked"] for b in G2["marked"]}
    return automaton(seen, all_events, trans, init, marked)

plant = compose(valve, bottle)
print(f"composed states: {len(plant['states'])}")
for s in sorted(plant["states"]):
    print("  ", s)

**Expectation.** The product has **4** states (the pairs of `{closed,open}`×`{no_bottle,bottle}`),
all reachable. Note the synchronisation worked: from `('open','bottle')` the shared event `depart`
is **not** enabled (the valve has no `depart` transition while `open`), so a bottle can never leave
with the valve open.

## Part E — a safety property, and why it fails (given)

The station's safety rule: **the valve must never be open when there is no bottle** (that would spray
product everywhere). The forbidden state is `('open', 'no_bottle')`. We check whether the
**uncontrolled** plant can reach it.

In [ ]:
FORBIDDEN = ("open", "no_bottle")
reach = reachable(plant)
print("all reachable product states:", len(reach))
print("forbidden state reachable?", FORBIDDEN in reach)

# show a shortest event sequence that reaches the forbidden state (BFS with paths)
def path_to(G, target):
    q = deque([(G["init"], [])]); seen = {G["init"]}
    while q:
        s, path = q.popleft()
        if s == target:
            return path
        for e in G["events"]:
            if (s, e) in G["trans"]:
                nxt = G["trans"][(s, e)]
                if nxt not in seen:
                    seen.add(nxt); q.append((nxt, path + [e]))
    return None

print("how it happens:", path_to(plant, FORBIDDEN))

**Expectation / self-check.** The forbidden state `('open','no_bottle')` **is reachable** — the
uncontrolled plant is **unsafe**: from the start, a single `open_valve` opens the valve with no bottle
present. Nothing in the plant prevents it, because `open_valve` is enabled regardless of the bottle.

This is exactly the problem **supervisory control** solves (P02): a supervisor would **disable the
controllable event `open_valve`** in any state with `no_bottle`, making the forbidden state
unreachable while allowing everything else. Note the asymmetry that makes it interesting: `arrive` is
*uncontrollable* (bottles come on their own), so the supervisor cannot simply forbid states by
blocking arrivals — it must act on the controllable events only.

## Conclusion

You have built the discrete-event foundation of automation:
- a **finite automaton** and its **simulation**,
- **reachability** by BFS — the engine of every safety check,
- **parallel composition** (the synchronous product) that combines components, with shared events synchronising,
- and a **safety analysis** showing the uncontrolled station is unsafe.

That unsafe finding is the bridge to **P02 (medium)**: model a resource-sharing cell as a **Petri
net**, find its **deadlocks**, and **synthesise a supervisor** that provably keeps the plant safe —
using only the controllable events, and as permissively as possible.